# Feature engineering for river flow forecasting

This notebook transforms raw 5-minute river level and flow readings into
daily-frequency features suitable for time-series forecasting. We resample
to daily means, create lag features, compute rolling statistics, perform
seasonal decomposition, and test for stationarity.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import fetch_river_data, preprocess, resample_daily, load_and_prepare

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and preprocess the raw data

`fetch_river_data` retrieves from the local CSV cache or the Calgary Open Data
API. `preprocess` parses timestamps, extracts temporal features, converts
level and flow_rate to numeric, and sorts chronologically.

In [ ]:
raw_df = fetch_river_data(limit=50000)
print(f'Raw records: {len(raw_df):,}')
print(f'Columns: {list(raw_df.columns)}')

In [ ]:
clean_df = preprocess(raw_df)
print(f'After preprocessing: {len(clean_df):,} rows')
print(f'Date range: {clean_df["timestamp"].min()} to {clean_df["timestamp"].max()}')
clean_df.head()

## 2. Resample to daily frequency

`resample_daily` aggregates 5-minute observations to daily means for level
and flow_rate. It then creates rolling averages (7-day and 30-day windows)
and lag features (1, 7, 14, and 30 days). Rows with NaN from the lag/rolling
computations are dropped.

In [ ]:
daily_df = resample_daily(clean_df)
print(f'Daily dataframe: {len(daily_df)} days, {daily_df.shape[1]} columns')
print(f'Columns: {list(daily_df.columns)}')
daily_df.head()

## 3. Visualise the daily time series

Plotting the full daily series shows seasonal cycles and any long-term trends.
River flow in Calgary peaks during spring snowmelt (May--June) and drops
in winter.

In [ ]:
target_col = 'flow_rate' if 'flow_rate' in daily_df.columns else 'level'
print(f'Target variable: {target_col}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_df['date'], daily_df[target_col], linewidth=0.7, color='steelblue')
ax.set_xlabel('Date')
ax.set_ylabel(target_col)
ax.set_title(f'Daily {target_col} over time', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Lag features

Lag features encode autoregressive information: the value 1 day ago, 7 days
ago, 14 days ago, and 30 days ago. These are the most direct predictors for
a time series that exhibits short-term persistence and weekly cycles.

In [ ]:
lag_cols = [c for c in daily_df.columns if '_lag_' in c]
print(f'Lag features: {lag_cols}')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, lag_cols[:4]):
    ax.scatter(daily_df[col], daily_df[target_col], s=2, alpha=0.3)
    r = daily_df[[col, target_col]].corr().iloc[0, 1]
    ax.set_xlabel(col)
    ax.set_ylabel(target_col)
    ax.set_title(f'{col} vs {target_col} (r={r:.3f})', fontsize=10)

plt.suptitle('Lag features vs target', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. Rolling means

The 7-day and 30-day rolling averages smooth out day-to-day noise and
capture short- and medium-term trends.

In [ ]:
rolling_cols = [c for c in daily_df.columns if 'rolling' in c]
print(f'Rolling features: {rolling_cols}')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_df['date'], daily_df[target_col], linewidth=0.5, alpha=0.5, label='Daily')
for col in rolling_cols:
    ax.plot(daily_df['date'], daily_df[col], linewidth=1.5, label=col)
ax.set_xlabel('Date')
ax.set_ylabel(target_col)
ax.set_title('Rolling averages overlaid on daily values', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Seasonal decomposition

We use `statsmodels.tsa.seasonal_decompose` to split the daily series into
trend, seasonal, and residual components. A period of 365 captures the
annual snowmelt cycle (if we have enough data).

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

series = daily_df.set_index('date')[target_col]
series.index = pd.to_datetime(series.index)

# Use period=30 if data covers less than one full year
period = min(365, len(series) // 2)
decomposition = seasonal_decompose(series, model='additive', period=period)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title='Observed')
decomposition.trend.plot(ax=axes[1], title='Trend')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
decomposition.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

## 7. Stationarity test (Augmented Dickey-Fuller)

Many time-series models assume stationarity. The ADF test checks whether
the series has a unit root. A p-value below 0.05 means we can reject the
null hypothesis of non-stationarity.

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(series.dropna(), autolag='AIC')

print('Augmented Dickey-Fuller test results:')
print(f'  Test statistic: {adf_result[0]:.4f}')
print(f'  p-value:        {adf_result[1]:.6f}')
print(f'  Lags used:      {adf_result[2]}')
print(f'  Observations:   {adf_result[3]}')
print('  Critical values:')
for key, val in adf_result[4].items():
    print(f'    {key}: {val:.4f}')

if adf_result[1] < 0.05:
    print('\nConclusion: the series is stationary (reject unit root).')
else:
    print('\nConclusion: the series is non-stationary (cannot reject unit root).')
    print('Differencing may be needed for ARIMA.')

In [ ]:
# Test stationarity of first difference
diff_series = series.diff().dropna()
adf_diff = adfuller(diff_series, autolag='AIC')
print(f'First-differenced ADF test: statistic={adf_diff[0]:.4f}, p-value={adf_diff[1]:.6f}')
if adf_diff[1] < 0.05:
    print('First difference is stationary.')

## 8. Feature correlation summary

In [ ]:
feature_cols = [c for c in daily_df.columns
                if any(tag in c for tag in ('lag_', 'rolling_', 'day_of_week', 'month', 'year'))]
corr_with_target = daily_df[feature_cols + [target_col]].corr()[target_col].drop(target_col)
corr_with_target = corr_with_target.abs().sort_values(ascending=False)

print(f'Feature correlations with {target_col} (absolute):\n')
print(corr_with_target.to_string())

## 9. Save processed daily data

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
daily_df.to_csv('../data/river_flow_daily_features.csv', index=False)
print(f'Saved: {len(daily_df)} days, {daily_df.shape[1]} columns')

## Summary

- Resampled 5-minute readings to daily means, reducing the dataset from
  tens of thousands of rows to a manageable daily series.
- Created lag features at 1, 7, 14, and 30 days. The 1-day lag has the
  strongest correlation with the target (typically r > 0.95).
- Rolling 7-day and 30-day means capture short- and medium-term trends.
- Seasonal decomposition confirms an annual cycle peaking during spring runoff.
- ADF test: the raw series may be non-stationary, but the first difference
  is stationary, supporting an ARIMA(p,1,q) specification.

Next: `03_modeling` trains Random Forest, XGBoost, and ARIMA models on
these features using a temporal train/test split.